# Погода в Москве и Санкт-Петербурге: сравнение средних и медиан

## 1. Вопрос

Какой из двух городов теплее, влажнее и ветренее, и **зависит ли ответ от того, чем мерить: средним или медианой?**

Почему это интересно. Годовая температура у Москвы и Петербурга близка, но устроена по-разному: в Москве климат более континентальный (холоднее зимой, жарче летом), в Петербурге его смягчает Балтика. Среднее чувствительно к экстремальным дням (морозы, жара, шторма), а медиана описывает «типичный» день. Если распределения у городов разной формы, среднее и медиана могут назвать разных «победителей».

Дополнительно проверю устойчивость: пересчитаю всё после удаления выбросов (правило 1,5 IQR внутри каждого месяца) и посмотрю, меняется ли вывод.

## 2. Данные

| | Москва | Санкт-Петербург |
|---|---|---|
| Источник | rp5.ru, «Расписание Погоды» | rp5.ru, «Расписание Погоды» |
| Станция | Москва (ВДНХ), WMO 27612 | Санкт-Петербург, WMO 26063 |
| Страница архива | https://rp5.ru/Архив_погоды_в_Москве_(ВДНХ) | https://rp5.ru/Архив_погоды_в_Санкт-Петербурге |
| Период | 01.01.2025 – 31.12.2025, все дни | 01.01.2025 – 31.12.2025, все дни |
| Формат | CSV, разделитель `;`, кодировка UTF-8 | CSV, разделитель `;`, кодировка UTF-8 |
| Дата скачивания | 13.11.2026 | 13.11.2026 |
| Объём | 2 920 строк | 2 920 строк |

* **Одна строка** = одно срочное наблюдение на метеостанции. Наблюдения идут каждые 3 часа (8 сроков в сутки, 365 × 8 = 2 920), время местное (московское).
* **Всего** 5 840 строк в двух файлах.
* **Столбцы, которые использую:** `T` температура воздуха на высоте 2 м (°C), `U` относительная влажность (%), `Ff` средняя скорость ветра (м/с), `Po` давление на уровне станции (мм рт. ст.), `RRR` количество осадков (мм), `tR` за сколько часов накоплены осадки.
* Файлы лежат в папке `data/` рядом с notebook под теми именами, под которыми их отдал rp5 (`27612.01.01.2025.31.12.2025.1.0.0.ru.utf8.00000000.csv` и такой же для 26063). Notebook сам находит их по индексу станции, открывает по относительному пути и ничего в них не меняет: вся очистка сделана кодом.
* Личных данных нет: только метеоизмерения.

In [1]:
import glob
import gzip
import io
import math
import os

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

# Город -> индекс ВМО. По индексу ищу скачанный с rp5 файл рядом с notebook
STATIONS = {'Москва': '27612', 'Санкт-Петербург': '26063'}

# Физически допустимые диапазоны: значения за их пределами считаю ошибкой записи
VALID_RANGES = {
    'T':  (-60, 45),    # °C
    'U':  (0, 100),     # %
    'Ff': (0, 50),      # м/с
    'Po': (650, 820),   # мм рт. ст.
}
NUM_COLS = ['T', 'U', 'Ff', 'Po']
MIN_OBS_PER_DAY = 6   # если за сутки меньше 6 из 8 сроков, среднее за день будет смещено, такой день не беру
IQR_K = 1.5           # множитель в правиле выбросов Q1 - k*IQR ... Q3 + k*IQR
PRECIP_HOURS = (9, 21)  # сроки измерения осадков по местному (московскому) времени = 06 и 18 UTC

Настройки заданы. Дальше загрузка: функция `load_table` находит файл станции, сама определяет кодировку (rp5 отдаёт UTF-8 или cp1251), пропускает служебные строки с `#` в начале файла и читает все значения **как текст**. Перевод в числа делаю позже, чтобы проверки увидели «сырые» проблемы: пустые ячейки и текст вместо чисел. Если файла нет, функция не падает, а пишет понятное сообщение и возвращает пустую таблицу.

In [2]:
def find_data_file(wmo_id, folders=('.', 'data')):
    # Ищет файл rp5 вида '<WMO>.<даты>...csv.gz' (или .csv) по относительному пути
    for folder in folders:
        found = sorted(glob.glob(os.path.join(folder, f'{wmo_id}.*.csv*')))
        if found:
            return found[0]
    return None


def read_text(path):
    # Читает .csv или .csv.gz; пробует UTF-8, потом cp1251. При неудаче возвращает None
    opener = gzip.open if path.endswith('.gz') else open
    try:
        with opener(path, 'rb') as f:
            raw_bytes = f.read()
    except OSError as err:
        print(f'Не удалось открыть {path}: {err}')
        return None
    for enc in ('utf-8-sig', 'cp1251'):
        try:
            return raw_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
    print(f'Не удалось определить кодировку файла {path}')
    return None


def load_table(wmo_id, city):
    empty = pd.DataFrame(columns=['time'] + NUM_COLS + ['RRR', 'tR'])
    path = find_data_file(wmo_id)
    if path is None:
        print(f'[{city}] Файл {wmo_id}.*.csv.gz не найден рядом с notebook. Скачайте архив с rp5.ru (раздел 2).')
        return empty
    text = read_text(path)
    if not text:
        return empty

    lines = text.splitlines()
    n_comment = 0
    while n_comment < len(lines) and lines[n_comment].startswith('#'):
        n_comment += 1
    meta = [line.lstrip('# ').strip() for line in lines[:n_comment] if line.strip('# ')]

    # Строки данных у rp5 оканчиваются ';', а заголовок может быть без неё. Тогда полей в данных
    # на одно больше, pandas делает время индексом, и все столбцы съезжают влево.
    # Поэтому убираю ';' в конце каждой строки и запрещаю индекс из первого столбца.
    body = '\n'.join(line.rstrip().rstrip(';') for line in lines[n_comment:] if line.strip())
    df = pd.read_csv(io.StringIO(body), sep=';', dtype=str, keep_default_na=False, index_col=False)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]      # пустой столбец после последней ';'
    df = df.rename(columns={df.columns[0]: 'time'})            # 'Местное время в ...' у каждой станции своё
    ok_time = pd.to_datetime(df['time'], format='%d.%m.%Y %H:%M', errors='coerce').notna().mean() if len(df) else 1
    if ok_time < 0.9:
        print(f'[{city}] ВНИМАНИЕ: первый столбец не похож на дату ({ok_time:.0%} строк разобрано). '
              f'Возможно, столбцы сдвинуты или выбран не тот формат файла.')
    for col in NUM_COLS + ['RRR', 'tR']:
        if col not in df.columns:
            print(f'[{city}] В файле нет столбца {col}: заполняю его пропусками')
            df[col] = ''
    print(f'[{city}] {path}: {len(df)} строк, {df.shape[1]} столбцов')
    if meta:
        print(f'    {meta[0]}')
    return df


raw = {city: load_table(wmo, city) for city, wmo in STATIONS.items()}
print(f'Всего строк в двух файлах: {sum(len(df) for df in raw.values())}')
display(raw['Москва'][['time'] + NUM_COLS + ['RRR', 'tR']].head())

[Москва] data/27612.01.01.2025.31.12.2025.1.0.0.ru.utf8.00000000.csv: 2920 строк, 29 столбцов
    Метеостанция Москва (ВДНХ), Россия, WMO_ID=27612, выборка с 01.01.2025 по 31.12.2025, все дни
[Санкт-Петербург] data/26063.01.01.2025.31.12.2025.1.0.0.ru.utf8.00000000.csv: 2920 строк, 29 столбцов
    Метеостанция Санкт-Петербург, Россия, WMO_ID=26063, выборка с 01.01.2025 по 31.12.2025, все дни
Всего строк в двух файлах: 5840


,time,T,U,Ff,Po,RRR,tR
0,31.12.2025 21:00,-7.9,86,1,731.3,0.3,12
1,31.12.2025 18:00,-7.2,82,1,730.5,0.8,12
2,31.12.2025 15:00,-7.0,79,1,729.5,,
3,31.12.2025 12:00,-6.6,84,1,728.8,,
4,31.12.2025 09:00,-6.2,87,2,728.2,2.0,12


**Что получилось.** Москва: 2 920 строк, Петербург: 2 920 строк, вместе 5 840, то есть требование «не меньше 200 строк» выполнено с большим запасом. 2 920 = 365 × 8, значит, за год не пропущен ни один срок наблюдений. Первая служебная строка файла подтверждает станцию и период. Строки в выгрузке rp5 идут от новых к старым, а в столбце `RRR` вместо чисел встречаются слова «Осадков нет» и «Следы осадков».

**Что это значит.** Данные нельзя сразу считать: время записано текстом, часть ячеек пустая (осадки измеряют не в каждый срок), а в осадках есть текстовые коды. Поэтому следующий шаг: проверить качество данных (пропуски, повторы, недопустимые значения), прежде чем что-то чистить.

## 3. Подготовка

In [3]:
def find_duplicates(keys):
    # Сколько ключей встречаются повторно (каждый повтор после первого считается один раз)
    seen, dups = set(), 0
    for key in keys:
        if key in seen:
            dups += 1
        else:
            seen.add(key)
    return dups


def check_data(df, city):
    report = {'город': city, 'строк': len(df)}
    if df.empty:
        print(f'[{city}] Данных нет, проверять нечего.')
        return report

    times = pd.to_datetime(df['time'], format='%d.%m.%Y %H:%M', errors='coerce')
    report['время не разобрано'] = int(times.isna().sum())
    report['повторы времени'] = find_duplicates(df['time'].tolist())
    for col in NUM_COLS:
        text = df[col].astype(str).str.strip()
        num = pd.to_numeric(text.str.replace(',', '.'), errors='coerce')
        lo, hi = VALID_RANGES[col]
        report[f'{col}: пусто'] = int((text == '').sum())
        report[f'{col}: не число'] = int(((text != '') & num.isna()).sum())
        report[f'{col}: вне [{lo}; {hi}]'] = int(((num < lo) | (num > hi)).sum())
    rrr = df['RRR'].astype(str).str.strip()
    report['RRR: пусто'] = int((rrr == '').sum())
    report['RRR: текстовые коды'] = int(rrr.isin(['Осадков нет', 'Следы осадков']).sum())

    problems = {k: v for k, v in report.items() if k not in ('город', 'строк') and v}
    if problems:
        print(f'[{city}] Найдено: ' + '; '.join(f'{k} = {v}' for k, v in problems.items()))
    else:
        print(f'[{city}] Проблем не найдено')
    return report


reports = [check_data(df, city) for city, df in raw.items()]
display(pd.DataFrame(reports).set_index('город').T)

[Москва] Найдено: Po: пусто = 1; RRR: пусто = 1461; RRR: текстовые коды = 900
[Санкт-Петербург] Найдено: RRR: пусто = 1453; RRR: текстовые коды = 889


город,Москва,Санкт-Петербург
строк,2920,2920
время не разобрано,0,0
повторы времени,0,0
T: пусто,0,0
T: не число,0,0
T: вне [-60; 45],0,0
U: пусто,0,0
U: не число,0,0
U: вне [0; 100],0,0
Ff: пусто,0,0


**Что получилось.** Данные с rp5 оказались чистыми. Время разобралось во всех строках, повторов сроков нет, ни одно значение температуры, влажности, ветра и давления не выходит за физические пределы. Пропусков в `T`, `U` и `Ff` нет; в Москве есть 1 пустое значение давления `Po`. Пусто в `RRR` 1 461 строка в Москве и 1 453 в Петербурге: это нормально, осадки записывают только в 4 из 8 сроков. Текстовых кодов («Осадков нет», «Следы осадков») 900 и 889.

**Что это значит и как чищу.** Проверки всё равно оставляю в конвейере: на другом периоде или станции проблемы могут появиться, и тогда код их обработает, а не упадёт.
* Строки с неразобранным временем и повторы одного и того же срока удаляю: повтор дал бы этому сроку двойной вес.
* Значения вне физических диапазонов **не удаляю вместе со строкой**, а заменяю на пропуск только в этом столбце: ошибка в давлении не значит, что температура в тот же срок неверна.
* «Осадков нет» = 0 мм; «Следы осадков» тоже 0 мм (меньше 0,1 мм, измерить нельзя); пустая ячейка = пропуск, а не ноль, иначе сроки без измерения выглядели бы сухими.

In [4]:
PRECIP_CODES = {'Осадков нет': 0.0, 'Следы осадков': 0.0}


def parse_precip(value):
    # Текст из столбца RRR -> число мм; пусто или мусор -> NaN
    if value is None:
        return math.nan
    s = str(value).strip()
    if s == '' or s.lower() == 'nan':
        return math.nan
    if s in PRECIP_CODES:
        return PRECIP_CODES[s]
    try:
        return float(s.replace(',', '.'))
    except ValueError:
        return math.nan


def clean(df, city):
    cols = ['time', 'date', 'month'] + NUM_COLS + ['RRR', 'tR']
    if df.empty:
        print(f'[{city}] Пустая таблица, очищать нечего.')
        return pd.DataFrame(columns=cols)

    out = df.copy()
    n0 = len(out)
    out['time'] = pd.to_datetime(out['time'], format='%d.%m.%Y %H:%M', errors='coerce')
    out = out.dropna(subset=['time'])
    n_bad_time = n0 - len(out)
    out = out.drop_duplicates(subset='time', keep='first')
    n_dup = n0 - n_bad_time - len(out)

    n_bad_value = 0
    for col in NUM_COLS:
        out[col] = pd.to_numeric(out[col].astype(str).str.strip().str.replace(',', '.'), errors='coerce')
        lo, hi = VALID_RANGES[col]
        bad = (out[col] < lo) | (out[col] > hi)
        n_bad_value += int(bad.sum())
        out.loc[bad, col] = np.nan
    out['RRR'] = out['RRR'].map(parse_precip).astype(float)
    out['tR'] = pd.to_numeric(out['tR'].astype(str).str.strip(), errors='coerce')

    out = out.sort_values('time').reset_index(drop=True)
    out['date'] = out['time'].dt.normalize()
    out['month'] = out['time'].dt.month
    print(f'[{city}] было {n0} строк -> осталось {len(out)} '
          f'(удалено: без времени {n_bad_time}, повторов {n_dup}); '
          f'недопустимых значений заменено на пропуск: {n_bad_value}')
    return out[cols]


clean_data = {city: clean(df, city) for city, df in raw.items()}
for city, df in clean_data.items():
    if not df.empty:
        print(f'[{city}] период {df["time"].min():%d.%m.%Y} – {df["time"].max():%d.%m.%Y}; '
              f'пропусков T: {df["T"].isna().sum()}, U: {df["U"].isna().sum()}, Ff: {df["Ff"].isna().sum()}')

[Москва] было 2920 строк -> осталось 2920 (удалено: без времени 0, повторов 0); недопустимых значений заменено на пропуск: 0
[Санкт-Петербург] было 2920 строк -> осталось 2920 (удалено: без времени 0, повторов 0); недопустимых значений заменено на пропуск: 0
[Москва] период 01.01.2025 – 31.12.2025; пропусков T: 0, U: 0, Ff: 0
[Санкт-Петербург] период 01.01.2025 – 31.12.2025; пропусков T: 0, U: 0, Ff: 0


**Что получилось.** После очистки в обоих городах осталось по 2 920 наблюдений: удалять было нечего, недопустимых значений 0. Период полный, с 01.01.2025 по 31.12.2025. Таблицы отсортированы по времени, температура, влажность, ветер и осадки стали числами.

**Почему дальше перехожу к суткам.** Сравнивать 3-часовые наблюдения напрямую неудобно: в них смешаны ночь и день, и пропуск одного срока сдвигает статистику. Естественная единица для вопроса «какой город теплее» это **сутки**: средняя за 8 сроков температура дня, минимум и максимум, средняя влажность и ветер, сумма осадков. Дни, где измерено меньше 6 сроков из 8, отбрасываю, иначе их среднее было бы смещено к той части суток, которая уцелела.

Тонкость с осадками: их измеряют за 12 часов в 09:00 и 21:00 по Москве, а rp5 повторяет ту же 12-часовую сумму ещё и в 06:00 и 18:00. Например, 15.07.2025 на ВДНХ 27 мм записаны и в 18:00, и в 21:00. Поэтому складываю только сроки 09:00 и 21:00.

In [5]:
def to_daily(df, city, min_obs=MIN_OBS_PER_DAY, verbose=True):
    cols = ['date', 'month', 'T_mean', 'T_min', 'T_max', 'U_mean', 'Ff_mean', 'precip', 'n_obs']
    if df.empty:
        if verbose:
            print(f'[{city}] Нет наблюдений, суточная таблица пустая.')
        return pd.DataFrame(columns=cols)

    g = df.groupby('date')
    daily = pd.DataFrame({
        'T_mean': g['T'].mean(), 'T_min': g['T'].min(), 'T_max': g['T'].max(),
        'U_mean': g['U'].mean(), 'Ff_mean': g['Ff'].mean(), 'n_obs': g['T'].count(),
    })
    # Осадки измеряют за 12 часов в сроки 06 и 18 UTC = 09:00 и 21:00 по Москве (tR = 12).
    # rp5 повторяет ту же 12-часовую сумму ещё и в соседних сроках (06:00 и 18:00),
    # поэтому беру только 09:00 и 21:00, иначе каждый дождь посчитается дважды.
    precip_rows = df[(df['tR'] == 12) & (df['time'].dt.hour.isin(PRECIP_HOURS))]
    if precip_rows.empty:
        precip_rows = df[df['tR'] == 12]
    daily['precip'] = precip_rows.groupby('date')['RRR'].sum(min_count=1).reindex(daily.index)
    daily = daily.reset_index()
    daily['month'] = daily['date'].dt.month

    short = daily['n_obs'] < min_obs
    if verbose:
        print(f'[{city}] суток: {len(daily)}, отброшено из-за неполных наблюдений: {int(short.sum())}, '
              f'в расчёт идёт: {int((~short).sum())}')
    return daily.loc[~short, cols].reset_index(drop=True)


daily = {city: to_daily(df, city) for city, df in clean_data.items()}
display(daily['Москва'].head())

[Москва] суток: 365, отброшено из-за неполных наблюдений: 0, в расчёт идёт: 365
[Санкт-Петербург] суток: 365, отброшено из-за неполных наблюдений: 0, в расчёт идёт: 365


,date,month,T_mean,T_min,T_max,U_mean,Ff_mean,precip,n_obs
0,2025-01-01,1,-1.9500,-4.1,0.8,88.625,1.000,5.0,8
1,2025-01-02,1,2.7500,1.4,3.9,85.000,1.875,1.0,8
2,2025-01-03,1,-2.0250,-5.4,2.8,79.125,1.000,2.0,8
3,2025-01-04,1,-2.9875,-3.7,-2.5,73.125,1.125,0.9,8
4,2025-01-05,1,-4.0625,-5.6,-2.8,82.250,1.125,1.1,8


**Что получилось.** В обоих городах в расчёт идут все 365 суток, неполных дней нет. Значит, сравнение городов идёт по одинаковому набору дней, и ни один город не получает преимущества из-за пропущенной зимы или лета.

**Что дальше.** Данные готовы, перехожу к расчётам. Для ключевых величин (средняя и медиана) использую собственные функции `my_mean` и `my_median`: они пропускают NaN и возвращают `None` на пустых данных вместо падения. Это те же функции, которые потом проверю на маленьких примерах в разделе 5.

## 4. Расчёты

In [6]:
def _valid(values):
    return [float(v) for v in values if v is not None and not pd.isna(v)]


def my_mean(values):
    vals = _valid(values)
    if not vals:
        return None
    return sum(vals) / len(vals)


def my_median(values):
    vals = sorted(_valid(values))
    n = len(vals)
    if n == 0:
        return None
    mid = n // 2
    return vals[mid] if n % 2 == 1 else (vals[mid - 1] + vals[mid]) / 2


def _r(x, nd=2):
    return None if x is None or pd.isna(x) else round(float(x), nd)


def compute_stats(day_df):
    t = _valid(day_df['T_mean']) if not day_df.empty else []
    u = day_df['U_mean'].tolist() if not day_df.empty else []
    ff = day_df['Ff_mean'].tolist() if not day_df.empty else []
    p = _valid(day_df['precip']) if not day_df.empty else []
    mean_t, med_t = my_mean(t), my_median(t)
    q25, q75 = (np.percentile(t, [25, 75]) if t else (None, None))
    return {
        'Дней в расчёте': len(day_df),
        'Средняя суточная температура, °C': _r(mean_t),
        'Медиана суточной температуры, °C': _r(med_t),
        'Среднее минус медиана (температура), °C': _r(mean_t - med_t) if t else None,
        'Стандартное отклонение температуры, °C': _r(np.std(t, ddof=1)) if len(t) > 1 else None,
        'Межквартильный размах температуры, °C': _r(q75 - q25) if t else None,
        'Самый холодный день, °C': _r(min(t)) if t else None,
        'Самый тёплый день, °C': _r(max(t)) if t else None,
        'Дней с морозом (минимум < 0 °C)': int((day_df['T_min'] < 0).sum()) if t else None,
        'Дней с жарой (максимум ≥ 25 °C)': int((day_df['T_max'] >= 25).sum()) if t else None,
        'Средняя влажность, %': _r(my_mean(u)),
        'Медиана влажности, %': _r(my_median(u)),
        'Средняя скорость ветра, м/с': _r(my_mean(ff)),
        'Медиана скорости ветра, м/с': _r(my_median(ff)),
        'Сумма осадков, мм': _r(sum(p), 1) if p else None,
        'Доля дней с осадками ≥ 1 мм, %': _r(100 * sum(x >= 1 for x in p) / len(p), 1) if p else None,
    }


# Пары «среднее / медиана», по которым отвечаю на главный вопрос
KEY_PAIRS = [
    ('температура', 'Средняя суточная температура, °C', 'Медиана суточной температуры, °C'),
    ('влажность', 'Средняя влажность, %', 'Медиана влажности, %'),
    ('ветер', 'Средняя скорость ветра, м/с', 'Медиана скорости ветра, м/с'),
]


def leader(stats_by_city, metric, tol=0.05):
    # Какой город выше по показателю и на сколько (Москва минус Петербург)
    (city_a, st_a), (city_b, st_b) = list(stats_by_city.items())[:2]
    va, vb = st_a.get(metric), st_b.get(metric)
    if va is None or vb is None:
        return 'нет данных', None
    diff = round(va - vb, 2)
    if abs(diff) < tol:
        return 'одинаково', diff
    return (city_a if diff > 0 else city_b), diff


full = {city: compute_stats(d) for city, d in daily.items()}
display(pd.DataFrame(full))

for label, mean_key, med_key in KEY_PAIRS:
    who_mean, d_mean = leader(full, mean_key)
    who_med, d_med = leader(full, med_key)
    print(f'{label:12s} по среднему выше: {who_mean} (М − П = {d_mean}); '
          f'по медиане выше: {who_med} (М − П = {d_med})')

,Москва,Санкт-Петербург
Дней в расчёте,365.00,365.00
"Средняя суточная температура, °C",8.51,8.02
"Медиана суточной температуры, °C",7.51,6.72
"Среднее минус медиана (температура), °C",0.99,1.29
"Стандартное отклонение температуры, °C",8.81,8.44
"Межквартильный размах температуры, °C",14.88,13.60
"Самый холодный день, °C",-12.16,-10.86
"Самый тёплый день, °C",27.88,26.81
Дней с морозом (минимум < 0 °C),95.00,98.00
Дней с жарой (максимум ≥ 25 °C),38.00,27.00


температура  по среднему выше: Москва (М − П = 0.49); по медиане выше: Москва (М − П = 0.79)
влажность    по среднему выше: Санкт-Петербург (М − П = -2.76); по медиане выше: Санкт-Петербург (М − П = -2.13)
ветер        по среднему выше: Санкт-Петербург (М − П = -0.79); по медиане выше: Санкт-Петербург (М − П = -0.87)


**Что получилось (16 показателей на город).**
* **Температура.** Средняя суточная 8,51 °C в Москве и 8,02 °C в Петербурге, медиана 7,51 и 6,72 °C. Москва теплее по обоим способам: на 0,49 °C по среднему и на 0,79 °C по медиане.
* **Разброс.** Стандартное отклонение 8,81 против 8,44 °C, межквартильный размах 14,88 против 13,60 °C. Разброс больше в Москве, что согласуется с более континентальным климатом.
* **Экстремумы.** Самый холодный день −12,16 °C в Москве и −10,86 °C в Петербурге, самый тёплый +27,88 и +26,81 °C. Дней с жарой (максимум от 25 °C) в Москве 38, в Петербурге 27. А дней с морозом почти поровну: 95 и 98.
* **Влажность** выше в Петербурге: среднее 77,7 против 75,0 %, медиана 79,4 против 77,3 %.
* **Ветер** в Петербурге почти вдвое сильнее: медиана 1,75 против 0,88 м/с.
* **Осадки** за год 658 мм в Москве и 764 мм в Петербурге; дней с осадками от 1 мм 31,8 % и 38,4 %.

**Что это значит.** В обоих городах среднее **выше** медианы (на 0,99 °C в Москве и на 1,29 °C в Петербурге). Больше половины дней 2025 года были холоднее «средней температуры»: тёплый сезон тянет среднее вверх, а зима 2025 года была мягкой (самый холодный день всего около −11…−12 °C), и сильных морозов, которые потянули бы среднее вниз, не было. Годовые цифры смешивают сезоны, поэтому дальше смотрю то же самое **по месяцам**: совпадает ли знак разницы «Москва минус Петербург» для среднего и медианы внутри каждого месяца.

In [7]:
def monthly_table(daily_by_city):
    cities = list(daily_by_city)
    if any(d.empty for d in daily_by_city.values()):
        print('Для помесячного сравнения нужны данные обоих городов.')
        return pd.DataFrame()
    table = pd.DataFrame(index=pd.Index(range(1, 13), name='месяц'))
    for city in cities:
        g = daily_by_city[city].groupby('month')['T_mean']
        table[f'{city}: среднее'] = g.apply(lambda s: _r(my_mean(s)))
        table[f'{city}: медиана'] = g.apply(lambda s: _r(my_median(s)))
    a, b = cities[:2]
    table['М − П по среднему'] = (table[f'{a}: среднее'] - table[f'{b}: среднее']).round(2)
    table['М − П по медиане'] = (table[f'{a}: медиана'] - table[f'{b}: медиана']).round(2)
    table['знак совпадает'] = np.sign(table['М − П по среднему']) == np.sign(table['М − П по медиане'])
    return table.dropna(how='all', subset=[c for c in table.columns if ':' in c])


monthly = monthly_table(daily)
display(monthly)
if not monthly.empty:
    print(f'Месяцев, где среднее и медиана называют разный «более тёплый» город: '
          f'{int((~monthly["знак совпадает"]).sum())} из {len(monthly)}')

,Москва: среднее,Москва: медиана,Санкт-Петербург: среднее,Санкт-Петербург: медиана,М − П по среднему,М − П по медиане,знак совпадает
месяц,,,,,,,
1,0.14,0.19,-0.57,0.11,0.71,0.08,True
2,-4.80,-5.68,-3.79,-3.46,-1.01,-2.22,True
3,4.19,4.57,2.74,2.10,1.45,2.47,True
4,8.81,8.77,6.03,6.27,2.78,2.50,True
5,13.20,12.24,10.84,11.18,2.36,1.06,True
6,16.94,16.58,15.50,15.39,1.44,1.19,True
7,21.64,21.21,21.58,22.57,0.06,-1.36,False
8,17.06,17.07,17.06,17.50,0.00,-0.43,False
9,13.75,14.72,14.91,15.81,-1.16,-1.09,True


Месяцев, где среднее и медиана называют разный «более тёплый» город: 2 из 12


**Что получилось.** По обоим способам Москва теплее Петербурга в 7 месяцах (январь, март–июнь, октябрь, ноябрь), сильнее всего весной: в апреле на 2,8 °C по среднему. Холоднее она в 3 месяцах: в феврале, сентябре и декабре. Среднее и медиана расходятся в 2 месяцах из 12, **в июле и августе**: по среднему Москва теплее на 0,06 °C в июле и равна Петербургу в августе (0,00 °C), а по медиане теплее Петербург (на 1,36 и 0,43 °C).

**Что это значит.** Когда разница между городами в месяце близка к нулю, ответ на вопрос «какой город теплее» зависит от выбранной статистики: в июле 2025 года в Петербурге было несколько прохладных дней, которые опустили среднее (21,58 °C при медиане 22,57 °C), а типичный июльский день был теплее именно в Петербурге. За год же перевес Москвы набирается весной и осенью, поэтому годовой вывод от этих двух месяцев не меняется. Следующий шаг: второй способ обработки. Уберу выбросы правилом 1,5 IQR **отдельно в каждом месяце** (иначе любая зима выглядела бы «выбросом» на фоне лета) и пересчитаю все показатели. Осадки не трогаю: сильный ливень это реальное событие, а правило IQR на ряде, где большинство значений 0, выбросило бы почти все дожди.

In [8]:
def iqr_bounds(values, k=IQR_K):
    vals = _valid(values)
    if len(vals) < 4:          # на 1–3 значениях квартили не имеют смысла
        return None
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def remove_outliers(df, city, cols=('T', 'U', 'Ff'), k=IQR_K, verbose=True):
    # Значения вне [Q1 - k*IQR; Q3 + k*IQR] внутри своего месяца -> NaN
    if df.empty:
        return df.copy(), {}
    out = df.copy()
    removed = {}
    for col in cols:
        n = 0
        for _, idx in out.groupby('month').groups.items():
            bounds = iqr_bounds(out.loc[idx, col], k)
            if bounds is None:
                continue
            values = out.loc[idx, col]
            mask = (values < bounds[0]) | (values > bounds[1])
            n += int(mask.sum())
            out.loc[mask[mask].index, col] = np.nan
        removed[col] = n
    if verbose:
        share = {c: f'{n} ({100 * n / max(out[c].notna().sum() + n, 1):.1f}%)' for c, n in removed.items()}
        print(f'[{city}] выбросов убрано: ' + ', '.join(f'{c}: {v}' for c, v in share.items()))
    return out, removed


trimmed_obs = {city: remove_outliers(df, city)[0] for city, df in clean_data.items()}
daily_trim = {city: to_daily(df, city) for city, df in trimmed_obs.items()}
trimmed = {city: compute_stats(d) for city, d in daily_trim.items()}
display(pd.DataFrame(trimmed))

[Москва] выбросов убрано: T: 14 (0.5%), U: 43 (1.5%), Ff: 132 (4.5%)
[Санкт-Петербург] выбросов убрано: T: 45 (1.5%), U: 51 (1.7%), Ff: 75 (2.6%)
[Москва] суток: 365, отброшено из-за неполных наблюдений: 2, в расчёт идёт: 363
[Санкт-Петербург] суток: 365, отброшено из-за неполных наблюдений: 7, в расчёт идёт: 358


,Москва,Санкт-Петербург
Дней в расчёте,363.00,358.00
"Средняя суточная температура, °C",8.61,7.98
"Медиана суточной температуры, °C",7.66,6.71
"Среднее минус медиана (температура), °C",0.95,1.27
"Стандартное отклонение температуры, °C",8.69,8.44
"Межквартильный размах температуры, °C",14.74,13.90
"Самый холодный день, °C",-10.75,-10.35
"Самый тёплый день, °C",27.88,26.81
Дней с морозом (минимум < 0 °C),93.00,96.00
Дней с жарой (максимум ≥ 25 °C),38.00,26.00


**Что получилось.** Правило IQR убрало в Москве 14 температурных наблюдений (0,5 %), 43 по влажности (1,5 %) и 132 по ветру (4,5 %); в Петербурге 45 (1,5 %), 51 (1,7 %) и 75 (2,6 %). После этого в Москве 2 дня, а в Петербурге 7 дней остались меньше чем с 6 сроками температуры и выпали из расчёта (363 и 358 суток).

**Что это значит.** Больше всего убрано по ветру в Москве: на ВДНХ ветер обычно слабый (около 1 м/с), поэтому любые 4–5 м/с правило считает выбросом. Убранные значения это реальная погода (самые холодные ночи месяца, порывы ветра), а не ошибки: ошибки уже искала проверка диапазонов в разделе 3. Поэтому «без выбросов» здесь не «более правильно», а другой способ посмотреть на данные. Теперь сведу оба способа в одну таблицу и проверю главное: меняется ли ответ на вопрос.

In [9]:
compare = pd.DataFrame({
    (city, method): pd.Series(st[city])
    for city in STATIONS
    for method, st in (('все данные', full), ('без выбросов', trimmed))
})
display(compare)

rows = []
for label, mean_key, med_key in KEY_PAIRS:
    for method, st in (('все данные', full), ('без выбросов', trimmed)):
        for stat_name, key in (('среднее', mean_key), ('медиана', med_key)):
            who, diff = leader(st, key)
            rows.append({'показатель': label, 'обработка': method, 'статистика': stat_name,
                         'выше у': who, 'М − П': diff})
verdicts = pd.DataFrame(rows)
display(verdicts.pivot_table(index='показатель', columns=['обработка', 'статистика'],
                             values='выше у', aggfunc='first', sort=False))

for label, _, _ in KEY_PAIRS:
    answers = sorted(set(verdicts.loc[verdicts['показатель'] == label, 'выше у']))
    if 'нет данных' in answers:
        print(f'{label}: нет данных для сравнения')
    elif len(answers) == 1:
        print(f'{label}: вывод устойчив, при всех 4 вариантах выше у «{answers[0]}»')
    else:
        print(f'{label}: вывод ЗАВИСИТ от способа, варианты ответа: {answers}')

for city in STATIONS:
    for label, mean_key, med_key in KEY_PAIRS[:1]:
        a, b = full[city][mean_key], trimmed[city][mean_key]
        c, d = full[city][med_key], trimmed[city][med_key]
        if None not in (a, b, c, d):
            print(f'{city}: без выбросов среднее сдвинулось на {b - a:+.2f} °C, медиана на {d - c:+.2f} °C')

Москва              Санкт-Петербург             
                                        все данные без выбросов      все данные без выбросов
Дней в расчёте                              365.00       363.00          365.00       358.00
Средняя суточная температура, °C              8.51         8.61            8.02         7.98
Медиана суточной температуры, °C              7.51         7.66            6.72         6.71
Среднее минус медиана (температура), °C       0.99         0.95            1.29         1.27
Стандартное отклонение температуры, °C        8.81         8.69            8.44         8.44
Межквартильный размах температуры, °C        14.88        14.74           13.60        13.90
Самый холодный день, °C                     -12.16       -10.75          -10.86       -10.35
Самый тёплый день, °C                        27.88        27.88           26.81        26.81
Дней с морозом (минимум < 0 °C)              95.00        93.00           98.00        96.00
Дней с жарой (максимум ≥ 25 °C)              38.00        38.00           27.00        26.00
Средняя влажность, %                         74.98        75.32           77.74        78.18
Медиана влажности, %                         77.25        77.62           79.38        80.12
Средняя скорость ветра, м/с                   1.01         0.98            1.80         1.77
Медиана скорости ветра, м/с                   0.88         0.88            1.75         1.62
Сумма осадков, мм                           658.40       657.90          764.40       764.40
Доля дней с осадками ≥ 1 мм, %               31.80        32.00           38.40        39.10

обработка         все данные                      без выбросов                 
статистика           среднее          медиана          среднее          медиана
показатель                                                                     
температура           Москва           Москва           Москва           Москва
влажность    Санкт-Петербург  Санкт-Петербург  Санкт-Петербург  Санкт-Петербург
ветер        Санкт-Петербург  Санкт-Петербург  Санкт-Петербург  Санкт-Петербург

температура: вывод устойчив, при всех 4 вариантах выше у «Москва»
влажность: вывод устойчив, при всех 4 вариантах выше у «Санкт-Петербург»
ветер: вывод устойчив, при всех 4 вариантах выше у «Санкт-Петербург»
Москва: без выбросов среднее сдвинулось на +0.10 °C, медиана на +0.15 °C
Санкт-Петербург: без выбросов среднее сдвинулось на -0.04 °C, медиана на -0.01 °C


**Что получилось.**
* **Температура: вывод устойчив.** Москва теплее во всех 4 вариантах. Разница «Москва минус Петербург» по среднему 0,49 °C на всех данных и 0,63 °C без выбросов; по медиане 0,79 и 0,95 °C.
* **Влажность: вывод устойчив**, выше в Петербурге во всех вариантах (разница от −2,1 до −2,9 процентного пункта).
* **Ветер: вывод устойчив**, сильнее в Петербурге во всех вариантах (на 0,74–0,87 м/с).
* После удаления выбросов средняя температура сдвинулась на +0,10 °C в Москве и −0,04 °C в Петербурге, медиана на +0,15 и −0,01 °C.

**Что это значит.** Ожидание «медиана устойчивее среднего» здесь проверить почти не на чем: оба сдвига не больше 0,15 °C, а в Москве медиана сдвинулась даже чуть сильнее среднего. Причина в том, что в Москве вместе с выбросами выпали 2 дня целиком, и это меняет середину ряда. Главное, что все сдвиги в несколько раз меньше разницы между городами, поэтому ответ не меняется. Прежде чем формулировать итог, проверю на маленьких примерах с заранее известным ответом, что сами функции считают верно и не падают на крайних случаях.

## 5. Проверка на маленьких примерах

Ответы посчитаны вручную заранее:
* среднее [2, 4, 6] = 12 / 3 = 4; среднее [1, NaN, 3] = 2 (NaN пропускается);
* медиана [1, 3, 2, 10] = (2 + 3) / 2 = 2,5; медиана одного значения [5] = 5;
* в [1, 2, 3, 4, 100]: Q1 = 2, Q3 = 4, IQR = 2, граница сверху 4 + 1,5·2 = 7, значит 100 выброс, а 1…4 нет;
* в ключах ['a', 'a', 'b', 'a'] повторов 2;
* сутки, где 1 мм записан в 06:00 и 09:00, а 2 мм в 18:00 и 21:00 (как повторяет rp5): осадков за сутки 1 + 2 = 3 мм, а не 6.

In [10]:
def run_checks():
    nan = float('nan')
    # 1. Среднее и медиана на обычных данных
    assert my_mean([2, 4, 6]) == 4
    assert my_mean([1, nan, 3]) == 2
    assert my_median([1, 3, 2, 10]) == 2.5
    # 2. Крайние случаи: пустые данные и одна строка
    assert my_mean([]) is None
    assert my_median([]) is None
    assert my_median([5]) == 5
    assert my_mean([nan, nan]) is None
    # 3. Повторы
    assert find_duplicates(['a', 'a', 'b', 'a']) == 2
    assert find_duplicates([]) == 0
    # 4. Разбор осадков rp5
    assert parse_precip('Осадков нет') == 0.0
    assert parse_precip('Следы осадков') == 0.0
    assert parse_precip('2.5') == 2.5
    assert math.isnan(parse_precip(''))
    assert math.isnan(parse_precip('ошибка'))
    # 5. Выбросы по IQR внутри месяца
    small = pd.DataFrame({'month': [1] * 5, 'T': [1.0, 2.0, 3.0, 4.0, 100.0]})
    cut, removed = remove_outliers(small, 'тест', cols=('T',), verbose=False)
    assert removed == {'T': 1} and math.isnan(cut.loc[4, 'T']) and cut['T'].notna().sum() == 4
    assert iqr_bounds([7]) is None                       # одна строка: границ нет, но и падения нет
    # 6. Весь конвейер на пустых данных и на одной строке
    empty_raw = pd.DataFrame(columns=['time'] + NUM_COLS + ['RRR', 'tR'])
    empty_stats = compute_stats(to_daily(clean(empty_raw, 'пусто'), 'пусто', verbose=False))
    assert empty_stats['Дней в расчёте'] == 0 and empty_stats['Средняя суточная температура, °C'] is None
    one_day = pd.DataFrame({'date': [pd.Timestamp('2025-01-01')], 'month': [1], 'T_mean': [-3.0],
                            'T_min': [-5.0], 'T_max': [-1.0], 'U_mean': [90.0], 'Ff_mean': [2.0],
                            'precip': [0.0], 'n_obs': [8]})
    one_stats = compute_stats(one_day)
    assert one_stats['Средняя суточная температура, °C'] == one_stats['Медиана суточной температуры, °C'] == -3.0
    assert one_stats['Стандартное отклонение температуры, °C'] is None   # по одной точке не считается
    assert one_stats['Дней с морозом (минимум < 0 °C)'] == 1
    # 7. Неверные значения: -999 вместо температуры превращается в пропуск, строка остаётся
    bad = pd.DataFrame({'time': ['01.01.2025 03:00', '01.01.2025 03:00', 'вчера'], 'T': ['-999', '-999', '1'],
                        'U': ['80', '80', '80'], 'Ff': ['2', '2', '2'], 'Po': ['750', '750', '750'],
                        'RRR': ['', '', ''], 'tR': ['', '', '']})
    cleaned = clean(bad, 'тест')
    assert len(cleaned) == 1 and cleaned['T'].isna().all()
    # 8. Осадки: rp5 повторяет 12-часовую сумму в соседнем сроке; за сутки должно быть 1 + 2 = 3 мм, а не 6
    hours = [0, 3, 6, 9, 12, 15, 18, 21]
    day = pd.DataFrame({'time': [pd.Timestamp(2025, 7, 15, h) for h in hours], 'T': [20.0] * 8,
                        'U': [70.0] * 8, 'Ff': [2.0] * 8, 'Po': [745.0] * 8,
                        'RRR': [np.nan, np.nan, 1.0, 1.0, np.nan, np.nan, 2.0, 2.0],
                        'tR': [np.nan, np.nan, 12, 12, np.nan, np.nan, 12, 12]})
    day['date'], day['month'] = day['time'].dt.normalize(), 7
    assert to_daily(day, 'тест', verbose=False).loc[0, 'precip'] == 3.0
    return 'Все проверки пройдены'


print(run_checks())

[пусто] Пустая таблица, очищать нечего.
[тест] было 3 строк -> осталось 1 (удалено: без времени 1, повторов 1); недопустимых значений заменено на пропуск: 1
Все проверки пройдены


**Что получилось.** Все проверки прошли. Функции дают те же ответы, что и ручной расчёт, а на пустых данных и на одной строке возвращают `None` или пустую таблицу вместо ошибки.

**Какие ошибки ловят и какие нет.** Проверки ловят неверную формулу среднего и медианы (в том числе при чётном числе значений), неправильную обработку NaN, текстовых кодов осадков, повторов и значений вроде −999, двойной счёт осадков из повторённых сроков rp5, а также падения на крайних входах. Они **не** ловят ошибки в самих исходных данных, которые выглядят правдоподобно (например, +12 °C, записанные вместо +21 °C), и не проверяют, что rp5 не изменит формат выгрузки. Расчёты подтверждены, сохраняю итоговые показатели в `results.txt`.

In [11]:
def save_results(full_stats, trimmed_stats, path='results.txt'):
    lines = []
    for method, st in (('все данные', full_stats), ('без выбросов', trimmed_stats)):
        for city, stats in st.items():
            for name, value in stats.items():
                lines.append(f'{city}, {method}, {name}: {"нет данных" if value is None else value}')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print(f'Записано {len(lines)} показателей в {path}')
    return lines


saved = save_results(full, trimmed)
print('\n'.join(saved[:8]))
print('...')

Записано 64 показателей в results.txt
Москва, все данные, Дней в расчёте: 365
Москва, все данные, Средняя суточная температура, °C: 8.51
Москва, все данные, Медиана суточной температуры, °C: 7.51
Москва, все данные, Среднее минус медиана (температура), °C: 0.99
Москва, все данные, Стандартное отклонение температуры, °C: 8.81
Москва, все данные, Межквартильный размах температуры, °C: 14.88
Москва, все данные, Самый холодный день, °C: -12.16
Москва, все данные, Самый тёплый день, °C: 27.88
...


## 6. Выводы

**Ответ на вопрос.**
* **Температура.** В 2025 году Москва была теплее Петербурга: средняя суточная температура 8,51 против 8,02 °C (+0,49 °C), медиана 7,51 против 6,72 °C (+0,79 °C). Вывод **устойчив**: он не меняется ни при замене среднего на медиану, ни при удалении выбросов (тогда разница 0,63 и 0,95 °C).
* **Где выбор статистики важен.** Внутри года среднее и медиана называют разный «более тёплый» город в 2 месяцах из 12, в июле и августе: по среднему города почти равны, а по медиане теплее Петербург (в июле на 1,36 °C). То есть при маленькой разнице между группами ответ может зависеть от того, чем мерить.
* **Форма распределения.** У Москвы больше разброс (стандартное отклонение 8,81 против 8,44 °C, межквартильный размах 14,88 против 13,60 °C) и больше жарких дней (38 против 27), что согласуется с более континентальным климатом. В обоих городах среднее выше медианы на 1,0–1,3 °C: больше половины дней холоднее «средней температуры».
* **Влажность, ветер, осадки.** Петербург влажнее (77,7 против 75,0 % по среднему), ветренее почти вдвое (медиана 1,75 против 0,88 м/с) и дождливее (764 против 658 мм, дней с осадками 38,4 против 31,8 %). Эти выводы тоже не меняются ни в одном из 4 вариантов.

**Ограничения.**
* Один год данных, причём нетипичный: зима 2025 года была мягкой (средняя за январь в Москве около 0 °C), поэтому выводы описывают 2025 год, а не климат. Для климатического вывода нужны 10 и больше лет.
* По одной станции на город: ВДНХ стоит в городской застройке (эффект «острова тепла»), а слабый ветер на ВДНХ частично объясняется её положением. Это сравнение двух точек, а не двух городов целиком.
* Суточные максимум и минимум взяты из 8 сроков наблюдений, поэтому настоящие экстремумы между сроками недооценены, и число жарких и морозных дней занижено.
* «Выбросы» по IQR это реальные морозы и порывы ветра, а не ошибки, поэтому вариант без выбросов показывает чувствительность выводов, а не «исправленные» данные.

## Как я использовал ИИ

1. Что просил у ИИ: Claude (Anthropic) помог с разбором формата выгрузки rp5 (функции `load_table` и `parse_precip`: служебные строки с `#`, кодировки, текстовые коды осадков) и с функцией `remove_outliers`, которая применяет правило IQR отдельно внутри каждого месяца.
2. Что проверил сам: посчитал вручную ответы для маленьких примеров в разделе 5 (среднее, медиана при чётном числе значений, границы IQR для [1, 2, 3, 4, 100]) и сверил с кодом; сверил число строк (365 × 8 = 2 920) и период с заголовками файлов; сравнил годовую сумму осадков с климатической нормой (около 700 мм).
3. Что поменял или переписал: при первом запуске на настоящих файлах все столбцы прочитались со сдвигом на один, потому что строки данных rp5 оканчиваются на `;`, а заголовок нет; исправил чтение файла. Ещё годовая сумма осадков вышла около 1 300 мм, почти вдвое выше нормы: оказалось, rp5 повторяет 12-часовую сумму в двух соседних сроках. Оставил только сроки 09:00 и 21:00 и добавил на это отдельную проверку в раздел 5.